# Vertex AI Gemini Batch Labeling & Cost/Token Attribution PoC

This interactive notebook walks through the end-to-end Proof of Concept (PoC) for **Batch Labeling Support for Gemini** on Vertex AI:

1. **Attach custom metadata labels (`labels`)** to a Gemini `batchPredictionJob` (`team=payments`, `env=prod`, `cost-center=4815`).
2. **Filter and organize batch jobs by label** via the Vertex AI REST API (`batchPredictionJobs.list` with `filter="labels.team=..."`).
3. **Inspect `GET batchPredictionJobs/{id}` upon `JOB_STATE_SUCCEEDED`** vs. per-row `usageMetadata` in the GCS JSONL output (`promptTokenCount`, `cachedContentTokenCount`, `candidatesTokenCount`, `thoughtsTokenCount`, `totalTokenCount`).
4. **Verify Cloud Monitoring (`token_count`) and Cloud Logging** dimensions and reporting latency.
5. **Inspect Vertex AI Gemini Batch SKUs** and execute the **BigQuery Cloud Billing Export SQL query** for centralized cost and token attribution.

## 1. User Configuration (Provide Your Project ID, Region, GCS Bucket, and Billing Export Table)

Set `PROJECT_ID`, `LOCATION`, `GCS_BUCKET`, and `BILLING_EXPORT_TABLE` below (or leave `PROJECT_ID = ""` to auto-detect from `GOOGLE_CLOUD_PROJECT` / Application Default Credentials).

In [ ]:
%load_ext google.cloud.bigquery

import datetime
import json
import os
import re
import time
from typing import Any, Dict, List, Optional

import google.auth
import google.auth.transport.requests
import requests
from google import genai
from google.cloud import storage

# ============================================================================
# USER CONFIGURATION: Provide your GCP Project ID, Region, GCS Bucket,
# and BigQuery Cloud Billing Export Table
# ============================================================================
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "<YOUR_PROJECT_ID>")
LOCATION = os.getenv("LOCATION_ID", "global")       # Gemini 3.8 requires 'global' endpoint
GCS_BUCKET_LOCATION = os.getenv("GCS_BUCKET_LOCATION", "us-central1")
GCS_BUCKET = os.getenv("POC_GCS_BUCKET", "<YOUR_GCS_BUCKET>")  # Auto-created if missing
BILLING_EXPORT_DATASET = os.getenv("BILLING_EXPORT_DATASET", "billing_export")
BILLING_EXPORT_TABLE = os.getenv(
    "BILLING_EXPORT_TABLE",
    "<BILLING_PROJECT>.<BILLING_DATASET>.gcp_billing_export_resource_v1_<BILLING_ACCOUNT_ID>",
)
MODEL_NAME = "gemini-3.8-flash"


def get_aiplatform_base_url(location: str, resource_name: Optional[str] = None) -> str:
    """Return the correct Vertex AI REST base URL for global vs. regional endpoints."""
    if resource_name and "/locations/global/" in resource_name:
        return "https://aiplatform.googleapis.com/v1"
    if location.lower() == "global":
        return "https://aiplatform.googleapis.com/v1"
    return f"https://{location}-aiplatform.googleapis.com/v1"


def get_auth_headers() -> tuple[Dict[str, str], Optional[str]]:
    """Obtain Bearer token headers and default project from Application Default Credentials."""
    creds, default_project = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)
    return {"Authorization": f"Bearer {creds.token}", "Content-Type": "application/json"}, default_project


def validate_gcp_labels(labels: Dict[str, str]) -> None:
    """Validate label keys/values against Google Cloud label constraints (max 64 labels)."""
    if len(labels) > 64:
        raise ValueError(f"Up to 64 labels per job allowed; got {len(labels)}")
    key_pattern = re.compile(r"^[a-z][a-z0-9_-]{0,62}$")
    val_pattern = re.compile(r"^[a-z0-9_-]{0,63}$")
    for k, v in labels.items():
        if not key_pattern.match(k):
            raise ValueError(f"Invalid GCP label key: {k!r}")
        if not val_pattern.match(v):
            raise ValueError(f"Invalid GCP label value: {v!r}")


headers, adc_project = get_auth_headers()
if not PROJECT_ID or PROJECT_ID == "<YOUR_PROJECT_ID>":
    PROJECT_ID = adc_project or ""

assert PROJECT_ID, "Please set PROJECT_ID above or export GOOGLE_CLOUD_PROJECT before running."

if not GCS_BUCKET or GCS_BUCKET == "<YOUR_GCS_BUCKET>":
    GCS_BUCKET = f"{PROJECT_ID}-gemini-batch-poc"

# Auto-detect Billing Account ID from Cloud Billing API if BILLING_EXPORT_TABLE is left as placeholder
b_info = requests.get(
    f"https://cloudbilling.googleapis.com/v1/projects/{PROJECT_ID}/billingInfo",
    headers=headers,
    timeout=30,
).json()
BILLING_ACCOUNT_NAME = b_info.get("billingAccountName", "")
BILLING_ACCOUNT_SUFFIX = (
    BILLING_ACCOUNT_NAME.split("/")[-1].replace("-", "_")
    if BILLING_ACCOUNT_NAME
    else "<BILLING_ACCOUNT_ID>"
)

if not BILLING_EXPORT_TABLE or "<BILLING_" in BILLING_EXPORT_TABLE:
    BILLING_EXPORT_TABLE = (
        f"{PROJECT_ID}.{BILLING_EXPORT_DATASET}.gcp_billing_export_resource_v1_{BILLING_ACCOUNT_SUFFIX}"
    )

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
JOB_LABELS = {
    "team": "payments",
    "env": "prod",
    "cost-center": "4815",
    "poc-run-id": RUN_ID,
    "traffic-mode": "batch",
}
validate_gcp_labels(JOB_LABELS)

print(f"Active Project ID    : {PROJECT_ID}")
print(f"Billing Account      : {BILLING_ACCOUNT_NAME} (billingEnabled={b_info.get('billingEnabled')})")
print(f"Gemini Model         : {MODEL_NAME} (Location: {LOCATION})")
print(f"GCS Staging Bucket   : {GCS_BUCKET} (auto-created in {GCS_BUCKET_LOCATION} if missing)")
print(f"Billing Export Table : {BILLING_EXPORT_TABLE}")
print(f"Job Labels           : {json.dumps(JOB_LABELS, indent=2)}")

## 2. Prepare & Upload Batch Input JSONL to GCS

We create a 2-row Gemini batch JSONL file in GCS:
* **Row 1**: Enables thinking (`thinkingBudget: 64`) so `thoughtsTokenCount` is generated alongside `promptTokenCount` and `candidatesTokenCount`.
* **Row 2**: Disables thinking (`thinkingBudget: 0`) for standard candidate token generation.

> **Important Note**: Labels are set at the **job level** (`BatchPredictionJob.labels` in Step 3), not inside individual JSONL `request` rows. Placing numeric-string labels like `"cost-center": "4815"` inside JSONL rows causes Vertex AI's JSONL parser to coerce `"4815"` into an integer `4815`, which fails protobuf map deserialization.

In [ ]:
assert GCS_BUCKET, "Please set GCS_BUCKET in Cell 2 before uploading batch input to GCS."

GCS_PREFIX = f"gemini-batch-labels-poc/{RUN_ID}"
OUTPUT_URI_PREFIX = f"gs://{GCS_BUCKET}/{GCS_PREFIX}/output"

batch_rows = [
    {
        "request": {
            "contents": [
                {
                    "role": "user",
                    "parts": [
                        {
                            "text": (
                                "In one short sentence, explain why cost attribution "
                                "matters for batch AI workloads."
                            )
                        }
                    ],
                }
            ],
            "generationConfig": {
                "temperature": 0.2,
                "maxOutputTokens": 128,
                "thinkingConfig": {"thinkingBudget": 64},
            },
        }
    },
    {
        "request": {
            "contents": [
                {
                    "role": "user",
                    "parts": [
                        {
                            "text": (
                                "List three key token types in Gemini usageMetadata "
                                "in 20 words or fewer."
                            )
                        }
                    ],
                }
            ],
            "generationConfig": {
                "temperature": 0.1,
                "maxOutputTokens": 64,
                "thinkingConfig": {"thinkingBudget": 0},
            },
        }
    },
]

jsonl_payload = "\n".join(json.dumps(r) for r in batch_rows) + "\n"
storage_client = storage.Client(project=PROJECT_ID)

bucket = storage_client.lookup_bucket(GCS_BUCKET)
if bucket is None:
    gcs_loc = GCS_BUCKET_LOCATION if LOCATION.lower() == "global" else LOCATION
    print(f"Bucket gs://{GCS_BUCKET} does not exist yet — creating in {gcs_loc}...")
    bucket = storage_client.create_bucket(GCS_BUCKET, project=PROJECT_ID, location=gcs_loc)
    print(f"Created bucket: gs://{bucket.name} ({bucket.location})")

blob_path = f"{GCS_PREFIX}/input/batch_requests.jsonl"
bucket.blob(blob_path).upload_from_string(jsonl_payload, content_type="application/jsonl")

INPUT_URI = f"gs://{GCS_BUCKET}/{blob_path}"
print(f"Uploaded {len(batch_rows)} requests to: {INPUT_URI}")
print(f"Batch output prefix configured as: {OUTPUT_URI_PREFIX}")

## 3. Create Labeled Gemini Batch Prediction Job (GA Capability #1)

In `google-genai` v1.x, `CreateBatchJobConfig` maps `display_name`, `dest`, and `webhook_config`. We pass `HttpOptions(extra_body={"labels": JOB_LABELS})` so the top-level `labels` map is set on the `BatchPredictionJob` resource at job creation time.

In [ ]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
display_name = f"gemini-3-8-batch-label-poc-{RUN_ID}"

batch_job = client.batches.create(
    model=MODEL_NAME,
    src=INPUT_URI,
    config=genai.types.CreateBatchJobConfig(
        display_name=display_name,
        dest=OUTPUT_URI_PREFIX,
        http_options=genai.types.HttpOptions(
            extra_body={"labels": JOB_LABELS}
        ),
    ),
)

BATCH_JOB_NAME = batch_job.name
print(f"Created Batch Job ({MODEL_NAME}): {BATCH_JOB_NAME}")
print(f"Initial State                  : {batch_job.state}")

headers, _ = get_auth_headers()
base_url = get_aiplatform_base_url(LOCATION, BATCH_JOB_NAME)
job_url = f"{base_url}/{BATCH_JOB_NAME}"
raw_job = requests.get(job_url, headers=headers, timeout=30).json()
print(f"Verified REST `BatchPredictionJob.labels`: {json.dumps(raw_job.get('labels', {}), indent=2)}")
print(f"Top-level keys on BatchPredictionJob: {sorted(raw_job.keys())}")

## 4. Run Online Labeled Call on Gemini

We also execute an online `generate_content` call on Gemini with custom `labels` so we can inspect its synchronous `usage_metadata` (`prompt_token_count`, `cached_content_token_count`, `candidates_token_count`, `thoughts_token_count`, `total_token_count`) and verify how `aiplatform.googleapis.com/publisher/online_serving/token_count` records those tokens in Cloud Monitoring.

In [ ]:
online_labels = dict(JOB_LABELS)
online_labels["traffic-mode"] = "online"

online_response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Explain in 2 sentences how Cloud Billing Export attributes Vertex AI costs.",
    config=genai.types.GenerateContentConfig(
        temperature=0.2,
        labels=online_labels,
    ),
)

usage = online_response.usage_metadata
print(f"Model               : {MODEL_NAME} (location={LOCATION})")
print(f"Online response text: {online_response.text.strip()}")
print(
    "Online synchronous usage_metadata:\n"
    f"  prompt_token_count         = {getattr(usage, 'prompt_token_count', 0) or 0}\n"
    f"  cached_content_token_count = {getattr(usage, 'cached_content_token_count', 0) or 0}\n"
    f"  candidates_token_count     = {getattr(usage, 'candidates_token_count', 0) or 0}\n"
    f"  thoughts_token_count       = {getattr(usage, 'thoughts_token_count', 0) or 0}\n"
    f"  total_token_count          = {getattr(usage, 'total_token_count', 0) or 0}"
)

## 5. Filter & Organize Batch Jobs by Custom Labels via API (GA Capability #2)

Verify that `batchPredictionJobs.list` supports filtering by custom labels (`labels.team="payments" AND labels.cost-center="4815"`).

In [ ]:
headers, _ = get_auth_headers()
filter_expr = f'labels.team="{JOB_LABELS["team"]}" AND labels.cost-center="{JOB_LABELS["cost-center"]}"'
base_url = get_aiplatform_base_url(LOCATION)
list_url = f"{base_url}/projects/{PROJECT_ID}/locations/{LOCATION}/batchPredictionJobs"

resp = requests.get(list_url, headers=headers, params={"filter": filter_expr}, timeout=30)
resp.raise_for_status()
all_matched_jobs = resp.json().get("batchPredictionJobs", [])
matched_jobs = [j for j in all_matched_jobs if MODEL_NAME in j.get("model", "")]

print(f"Filter Expression : {filter_expr}")
print(f"Matched {MODEL_NAME} Jobs Count: {len(matched_jobs)}")
for j in matched_jobs[:5]:
    print(f" - {j.get('name')} | model={j.get('model')} | state={j.get('state')} | labels={j.get('labels')}")

## 6. Inspect `GET batchPredictionJobs/{id}` vs. Per-Row GCS JSONL `usageMetadata`

This cell checks `GET batchPredictionJobs/{id}` (showing that `completionStats` and `labels` are present, while `usageMetadata` is **not** on the job resource itself). Once `state == JOB_STATE_SUCCEEDED`, it reads the GCS output `predictions.jsonl`, sums `usageMetadata` across rows, and plots the token breakdown.

In [ ]:
import matplotlib.pyplot as plt

# Optional: Override TARGET_JOB_NAME to inspect a specific batch job
TARGET_JOB_NAME = globals().get("BATCH_JOB_NAME", "")
POLL_SECONDS = 5  # Increase if you want the cell to wait until the new job reaches JOB_STATE_SUCCEEDED

assert TARGET_JOB_NAME, "Set TARGET_JOB_NAME or run Step 3 first."

headers, _ = get_auth_headers()
base_url = get_aiplatform_base_url(LOCATION, TARGET_JOB_NAME)
job_url = f"{base_url}/{TARGET_JOB_NAME}"
deadline = time.time() + POLL_SECONDS

while True:
    raw_job = requests.get(job_url, headers=headers, timeout=30).json()
    state = raw_job.get("state", "")
    print(f"[{MODEL_NAME}] Job state: {state} | completionStats: {raw_job.get('completionStats')}")
    if state in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED") or time.time() >= deadline:
        break
    time.sleep(5)

# If the newly submitted job is still queued, inspect the most recent gemini-3.8-flash batch job with completed predictions in GCS
sc = storage.Client(project=PROJECT_ID)
candidates_to_check = [raw_job] + globals().get("matched_jobs", [])
inspected_job = raw_job
prediction_blobs = []

for cand in candidates_to_check:
    gcs_out = (
        cand.get("outputInfo", {}).get("gcsOutputDirectory")
        or cand.get("outputConfig", {}).get("gcsDestination", {}).get("outputUriPrefix")
        or ""
    )
    if gcs_out.startswith("gs://"):
        bkt_name, _, prefix = gcs_out[5:].partition("/")
        blobs = [b for b in sc.list_blobs(bkt_name, prefix=prefix) if b.name.endswith("predictions.jsonl")]
        if blobs:
            inspected_job = cand
            prediction_blobs = blobs
            break

print(f"\nInspecting Job Resource : {inspected_job.get('name')}")
print("Does GET batchPredictionJobs/{id} contain `usageMetadata`?", "usageMetadata" in inspected_job)
print("Fields on GET batchPredictionJobs/{id}:", sorted(inspected_job.keys()))
print("completionStats on job resource:", json.dumps(inspected_job.get("completionStats")))
print("labels on job resource:", json.dumps(inspected_job.get("labels"), indent=2))

totals = {
    "promptTokenCount": 0,
    "cachedContentTokenCount": 0,
    "candidatesTokenCount": 0,
    "thoughtsTokenCount": 0,
    "totalTokenCount": 0,
}

if prediction_blobs:
    for blob in prediction_blobs:
        print(f"\nReading GCS Batch Output: gs://{blob.bucket.name}/{blob.name}")
        for idx, line in enumerate(blob.download_as_text().splitlines(), 1):
            if line.strip():
                row_obj = json.loads(line)
                resp_obj = row_obj.get("response", {})
                u = resp_obj.get("usageMetadata", {})
                print(f" - Row {idx} ({resp_obj.get('modelVersion')}): usageMetadata = {json.dumps(u)}")
                for k in totals:
                    totals[k] += int(u.get(k, 0))
    print("\nAggregated Per-Row usageMetadata from Gemini 3.8 Batch GCS JSONL:")
    print(json.dumps(totals, indent=2))
    chart_title = f"Gemini 3.8 Batch Job Token Breakdown (Total: {totals['totalTokenCount']} tokens)"
else:
    totals = {
        "promptTokenCount": getattr(usage, "prompt_token_count", 0) or 0,
        "cachedContentTokenCount": getattr(usage, "cached_content_token_count", 0) or 0,
        "candidatesTokenCount": getattr(usage, "candidates_token_count", 0) or 0,
        "thoughtsTokenCount": getattr(usage, "thoughts_token_count", 0) or 0,
        "totalTokenCount": getattr(usage, "total_token_count", 0) or 0,
    }
    print(json.dumps(totals, indent=2))
    chart_title = f"Gemini 3.8 usageMetadata Token Breakdown (Total: {totals['totalTokenCount']} tokens)"

categories = ["Prompt Input", "Cached Input", "Candidate Output", "Thinking Tokens"]
values = [
    totals["promptTokenCount"],
    totals["cachedContentTokenCount"],
    totals["candidatesTokenCount"],
    totals["thoughtsTokenCount"],
]
colors = ["#4285F4", "#34A853", "#FBBC05", "#EA4335"]
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(categories, values, color=colors)
ax.set_title(chart_title)
ax.set_ylabel("Token Count")
ax.set_ylim(0, max(values + [10]) * 1.2)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(values + [10]) * 0.02, str(val), ha="center")
plt.tight_layout()
plt.show()

## 7. Check Cloud Monitoring (`token_count`) & Cloud Logging Dimensions

Inspect the live `aiplatform.googleapis.com/publisher/online_serving/token_count` metric descriptor and recent time series in your project (`source="batch_..."` vs. online) to verify supported metric dimensions.

In [ ]:
headers, _ = get_auth_headers()

md_url = (
    f"https://monitoring.googleapis.com/v3/projects/{PROJECT_ID}/metricDescriptors/"
    "aiplatform.googleapis.com/publisher/online_serving/token_count"
)
md = requests.get(md_url, headers=headers, timeout=30).json()
label_keys = [lbl.get("key") for lbl in md.get("labels", [])]
print("Metric Descriptor: aiplatform.googleapis.com/publisher/online_serving/token_count")
print("  Monitored Resource Types:", md.get("monitoredResourceTypes"))
print("  Supported Metric Labels :", label_keys)
print("  Contains job_id or custom user label dimension?", any("label" in k or "job" in k for k in label_keys))

now = datetime.datetime.now(datetime.timezone.utc)
start = (now - datetime.timedelta(hours=24)).strftime("%Y-%m-%dT%H:%M:%SZ")
end = now.strftime("%Y-%m-%dT%H:%M:%SZ")
ts_url = f"https://monitoring.googleapis.com/v3/projects/{PROJECT_ID}/timeSeries"
ts_resp = requests.get(
    ts_url,
    headers=headers,
    params={
        "filter": (
            'metric.type = "aiplatform.googleapis.com/publisher/online_serving/token_count" '
            f'AND resource.labels.model_user_id = "{MODEL_NAME}"'
        ),
        "interval.startTime": start,
        "interval.endTime": end,
    },
    timeout=30,
)
series = ts_resp.json().get("timeSeries", [])
print(f"\nRecent `{MODEL_NAME}` `token_count` TimeSeries in last 24h: {len(series)} series found")
for ts in series[:10]:
    m_labels = ts.get("metric", {}).get("labels", {})
    r_labels = ts.get("resource", {}).get("labels", {})
    pts = sum(int(p.get("value", {}).get("int64Value", 0)) for p in ts.get("points", []))
    print(
        f" - model={r_labels.get('model_user_id')} | source={m_labels.get('source')} | "
        f"type={m_labels.get('type')} | explicit_caching={m_labels.get('explicit_caching')} | "
        f"sum_tokens={pts} | metric_labels={m_labels}"
    )

## 8. Inspect Vertex AI Gemini Batch & Context Caching SKUs

Query the Cloud Billing Catalog API (`services/C7E2-9256-1C43/skus`) to list the **Gemini** Batch Prediction and Context Caching SKUs in Vertex AI.

In [ ]:
headers, _ = get_auth_headers()
skus_url = "https://cloudbilling.googleapis.com/v1/services/C7E2-9256-1C43/skus"
skus_38: List[Dict[str, Any]] = []
page_token: Optional[str] = None

while True:
    params: Dict[str, Any] = {"pageSize": 5000}
    if page_token:
        params["pageToken"] = page_token
    resp = requests.get(skus_url, headers=headers, params=params, timeout=30)
    if resp.status_code != 200:
        break
    data = resp.json()
    for sku in data.get("skus", []):
        desc = sku.get("description", "")
        dl = desc.lower()
        if "gemini 3.8 flash" in dl and ("batch" in dl or "caching" in dl):
            skus_38.append(sku)
    page_token = data.get("nextPageToken")
    if not page_token:
        break

print(f"Gemini 3.8 Flash Batch & Caching SKUs ({len(skus_38)} found):")
for s in sorted(skus_38, key=lambda x: x.get("description", "")):
    if "text" in s.get("description", "").lower():
        print(f" - {s.get('skuId')}: {s.get('description')}")

## 9. Cloud Billing Export BigQuery Attribution Query (`%%bigquery` Magic)

Uses the Jupyter `%%bigquery` cell magic (`%load_ext google.cloud.bigquery`) to execute the BigQuery Cloud Billing Export attribution query directly against `bipin-dev-project-465919.billing_export.gcp_billing_export_resource_v1_01C5BC_7175D6_6BE083` and store the resulting DataFrame in `df_billing`.

In [ ]:
%%bigquery df_billing --project bipin-dev-project-465919
SELECT
  DATE(usage_start_time) AS usage_date,
  project.id AS project_id,
  sku.id AS sku_id,
  sku.description AS sku_description,
  (SELECT l.value FROM UNNEST(labels) AS l WHERE l.key = 'team') AS team,
  (SELECT l.value FROM UNNEST(labels) AS l WHERE l.key = 'env') AS env,
  (SELECT l.value FROM UNNEST(labels) AS l WHERE l.key = 'cost-center') AS cost_center,
  (SELECT l.value FROM UNNEST(labels) AS l WHERE l.key = 'poc-run-id') AS poc_run_id,
  (SELECT l.value FROM UNNEST(labels) AS l WHERE l.key = 'datacloud') AS datacloud_label,
  TO_JSON_STRING(labels) AS all_labels_json,
  CASE
    WHEN LOWER(sku.description) LIKE '%caching%' AND LOWER(sku.description) NOT LIKE '%storage%'
      THEN 'cached_input_tokens'
    WHEN LOWER(sku.description) LIKE '%input%' AND LOWER(sku.description) NOT LIKE '%caching%'
      THEN 'prompt_input_tokens'
    WHEN LOWER(sku.description) LIKE '%thinking%'
      THEN 'thinking_output_tokens'
    WHEN LOWER(sku.description) LIKE '%output%'
      THEN 'output_tokens (candidate_plus_thinking)'
    ELSE 'other'
  END AS token_category,
  usage.unit AS usage_unit,
  SUM(usage.amount) AS total_tokens,
  ROUND(SUM(cost) + SUM(IFNULL((SELECT SUM(c.amount) FROM UNNEST(credits) AS c), 0)), 6) AS net_cost_usd
FROM
  `bipin-dev-project-465919.billing_export.gcp_billing_export_resource_v1_01C5BC_7175D6_6BE083`
WHERE
  service.id = 'C7E2-9256-1C43' -- Vertex AI
  AND project.id = 'bipin-dev-project-465919'
  AND usage_start_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 35 DAY)
  -- Uncomment once initial chronological backfill reaches today's date:
  -- AND EXISTS (SELECT 1 FROM UNNEST(labels) AS l WHERE l.key = 'team' AND l.value = 'payments')
GROUP BY
  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12
ORDER BY
  usage_date DESC, total_tokens DESC

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

print(f"Returned {len(df_billing)} Vertex AI Batch Prediction rows from Cloud Billing Export:")
display(df_billing)